# Statistical Analysis and Predictive Modeling of Influencer Marketing ROI
## Using Multiple Linear Regression (Python + Statsmodels)

---

**Project Title:** Predictive Statistical Analysis of Influencer Marketing ROI Using Multiple Linear Regression

**Dataset:** Real-world Influencer Marketing ROI Dataset — 150,000 campaigns (Kaggle)

**Technologies Used:** Python, Pandas, NumPy, Matplotlib, Seaborn, Statsmodels, Excel

**Objective:** Build a statistical regression model to predict ROI of influencer marketing campaigns

---

### Table of Contents
1. Import Libraries
2. Load Dataset
3. Data Cleaning
4. Feature Engineering (ROI, Engagement Rate, Campaign Cost, Revenue)
5. Exploratory Data Analysis (EDA)
6. Statistical Analysis
7. Correlation Analysis
8. Multiple Linear Regression (Statsmodels)
9. Model Interpretation
10. Residual Analysis
11. Prediction for New Campaign
12. Model Evaluation Summary
13. Conclusion

---
## Module 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('All libraries imported successfully.')

---
## Module 2 — Load Dataset
Loading the real-world Kaggle dataset: **150,000 influencer marketing campaigns** collected across Instagram, YouTube, TikTok and Twitter (2022–2024).

In [ ]:
df = pd.read_csv('../Dataset/influencer_marketing_roi_dataset.csv')

print(f'Dataset loaded successfully!')
print(f'Shape : {df.shape[0]:,} rows x {df.shape[1]} columns')
print()
print('Column Names and Data Types:')
print(df.dtypes)
print()
print('First 5 Rows:')
df.head()

In [ ]:
print('Last 5 Rows:')
df.tail()

---
## Module 3 — Data Cleaning
Checking data quality before any analysis.

In [ ]:
# ── Missing Values ────────────────────────────────────────────────────────────
print('Missing Values per Column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# ── Duplicates ────────────────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')
if dupes > 0:
    df = df.drop_duplicates()
    print(f'Removed. New shape: {df.shape}')
else:
    print('No duplicates found.')

In [ ]:
# ── Invalid Values ────────────────────────────────────────────────────────────
print(f'Rows with engagements <= 0  : {(df["engagements"] <= 0).sum()}')
print(f'Rows with estimated_reach<=0: {(df["estimated_reach"] <= 0).sum()}')
print(f'Rows with product_sales < 0 : {(df["product_sales"] < 0).sum()}')

# Remove rows where reach is zero to avoid division errors
df = df[df['estimated_reach'] > 0].copy()
print(f'\nClean dataset shape: {df.shape}')

In [ ]:
# ── Parse Dates ───────────────────────────────────────────────────────────────
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date']   = pd.to_datetime(df['end_date'], errors='coerce')
df = df.dropna(subset=['start_date']).copy()
df = df.reset_index(drop=True)
# Extract month only (year values are not meaningful in this dataset)
df['month'] = df['start_date'].dt.month
print(f'Rows after date cleaning: {len(df):,}')
df['month'] = df['start_date'].dt.month

print('Date range of campaigns:')
print(f'  From : {df["start_date"].min().date()}')
print(f'  To   : {df["end_date"].dropna().max().date()}')

---
## Module 4 — Feature Engineering

The Kaggle dataset does not directly contain `Campaign_Cost`, `Revenue`, or `ROI`.
We engineer these columns using standard industry formulas.

| Engineered Column | Formula / Logic |
|-------------------|-----------------|
| **Engagement_Rate** | `engagements / estimated_reach × 100` |
| **Campaign_Cost** | `estimated_reach × CPM / 1000` where CPM varies by platform |
| **Revenue** | `product_sales × avg_order_value` (avg ₹40 per unit — calibrated to realistic ROI range) |
| **ROI** | `(Revenue − Campaign_Cost) / Campaign_Cost` |

**CPM (Cost Per 1000 impressions) by platform — industry standard rates (INR):**
- Instagram : ₹150
- YouTube   : ₹120
- TikTok    : ₹100
- Twitter   : ₹80

In [ ]:
# ── 1. Engagement Rate ────────────────────────────────────────────────────────
df['engagement_rate'] = (df['engagements'] / df['estimated_reach'] * 100).round(4)
df['engagement_rate'] = df['engagement_rate'].clip(upper=100)  # cap at 100%

# ── 2. Campaign Cost using platform CPM ──────────────────────────────────────
cpm_map = {'Instagram': 150, 'YouTube': 120, 'TikTok': 100, 'Twitter': 80}
df['cpm'] = df['platform'].map(cpm_map).fillna(100)
df['campaign_cost'] = ((df['estimated_reach'] / 1000) * df['cpm']).round(2)

# ── 3. Revenue = product_sales × average order value ────────────────────────
# Calibrated so that avg ROI sits in a realistic 0–5 range
# avg campaign_cost ≈ ₹60,000 | avg product_sales ≈ 2500 → AOV ≈ ₹40 gives Revenue ≈ ₹100,000
AVG_ORDER_VALUE = 40
df['revenue'] = (df['product_sales'] * AVG_ORDER_VALUE).round(2)

# ── 4. ROI = (Revenue - Cost) / Cost ─────────────────────────────────────────
df['roi'] = ((df['revenue'] - df['campaign_cost']) / df['campaign_cost']).round(4)

# Remove rows where campaign_cost = 0 (edge case)
df = df[df['campaign_cost'] > 0].copy()

print('Feature Engineering complete.')
print(f'Shape after engineering: {df.shape}')
print()
print('New columns added:')
print(df[['engagement_rate','campaign_cost','revenue','roi']].describe().round(4))

In [ ]:
# ── Show sample engineered rows ───────────────────────────────────────────────
print('Sample rows with engineered columns:')
df[['campaign_id','platform','engagements','estimated_reach',
    'product_sales','engagement_rate','campaign_cost','revenue','roi']].head(8)

In [ ]:
# ── ROI sanity check ──────────────────────────────────────────────────────────
print('ROI Distribution Check:')
print(f'  Min ROI  : {df["roi"].min():.4f}')
print(f'  Max ROI  : {df["roi"].max():.4f}')
print(f'  Mean ROI : {df["roi"].mean():.4f}')
print(f'  Campaigns with ROI > 0 (profitable): {(df["roi"]>0).sum():,} ({(df["roi"]>0).mean()*100:.1f}%)')
print(f'  Campaigns with ROI < 0 (loss)       : {(df["roi"]<0).sum():,} ({(df["roi"]<0).mean()*100:.1f}%)')

# Drop extreme outliers beyond 3 std for cleaner regression
roi_mean = df['roi'].mean()
roi_std  = df['roi'].std()
df = df[(df['roi'] > roi_mean - 3*roi_std) & (df['roi'] < roi_mean + 3*roi_std)].copy()
print(f'\nAfter removing ROI outliers (>3 std): {len(df):,} rows remaining')

---
## Module 5 — Exploratory Data Analysis (EDA)
Visualising the data before modeling to understand distributions, trends, and outliers.

In [ ]:
# Use a 10K sample for all heavy visualisations — statistically valid
PLOT_SAMPLE = df.sample(10000, random_state=42) if len(df) > 10000 else df

# ── 1. ROI Distribution ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(PLOT_SAMPLE['roi'], bins=60, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].axvline(df['roi'].mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean={df["roi"].mean():.2f}')
axes[0].axvline(df['roi'].median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median={df["roi"].median():.2f}')
axes[0].set_title('ROI Distribution (Histogram)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('ROI')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sns.kdeplot(PLOT_SAMPLE['roi'], ax=axes[1], fill=True, color='steelblue', linewidth=2)
axes[1].set_title('ROI Density Curve (KDE)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('ROI')
axes[1].set_ylabel('Density')

plt.suptitle('ROI Distribution Analysis — 150,000 Campaigns', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/01_roi_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: 01_roi_distribution.png')

In [ ]:
# ── 2. Campaign Count and ROI by Platform ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plat_counts = df['platform'].value_counts()
axes[0].bar(plat_counts.index, plat_counts.values,
             color=['#E1306C','#FF0000','#010101','#1DA1F2'], edgecolor='grey', linewidth=0.5)
axes[0].set_title('Number of Campaigns by Platform', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Platform')
axes[0].set_ylabel('Number of Campaigns')
for i, v in enumerate(plat_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)

sns.boxplot(data=PLOT_SAMPLE, x='platform', y='roi', palette='Set2', ax=axes[1])
axes[1].axhline(0, color='red', linestyle='--', linewidth=1, label='Break-even')
axes[1].set_title('ROI Distribution by Platform', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Platform')
axes[1].set_ylabel('ROI')
axes[1].legend()

plt.suptitle('Platform Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/02_platform_analysis.png', bbox_inches='tight')
plt.show()
print('Saved: 02_platform_analysis.png')

In [ ]:
# ── 3. ROI by Influencer Category ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cat_roi = PLOT_SAMPLE.groupby('influencer_category')['roi'].mean().sort_values(ascending=False)
axes[0].barh(cat_roi.index, cat_roi.values,
              color=['#2ecc71' if v > 0 else '#e74c3c' for v in cat_roi.values],
              edgecolor='grey', linewidth=0.4)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Average ROI by Influencer Category', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Average ROI')

sns.boxplot(data=PLOT_SAMPLE, x='influencer_category', y='roi', palette='pastel', ax=axes[1])
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_title('ROI Distribution by Category', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Influencer Category')
axes[1].set_ylabel('ROI')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Influencer Category Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/03_category_analysis.png', bbox_inches='tight')
plt.show()
print('Saved: 03_category_analysis.png')

In [ ]:
# ── 4. Scatter Plots ──────────────────────────────────────────────────────────
# Sample 5000 rows for scatter readability
sample = df.sample(5000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(sample['engagement_rate'], sample['roi'],
                 alpha=0.3, color='coral', edgecolors='none', s=10)
m, b = np.polyfit(sample['engagement_rate'], sample['roi'], 1)
x_line = np.linspace(sample['engagement_rate'].min(), sample['engagement_rate'].max(), 100)
axes[0].plot(x_line, m*x_line + b, color='darkred', linewidth=2, label=f'Trend (slope={m:.3f})')
axes[0].set_title('Engagement Rate vs ROI', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Engagement Rate (%)')
axes[0].set_ylabel('ROI')
axes[0].legend()

axes[1].scatter(np.log10(sample['campaign_cost']+1), sample['roi'],
                 alpha=0.3, color='steelblue', edgecolors='none', s=10)
m2, b2 = np.polyfit(np.log10(sample['campaign_cost']+1), sample['roi'], 1)
x_line2 = np.linspace(np.log10(sample['campaign_cost']+1).min(),
                       np.log10(sample['campaign_cost']+1).max(), 100)
axes[1].plot(x_line2, m2*x_line2 + b2, color='navy', linewidth=2, label=f'Trend (slope={m2:.3f})')
axes[1].set_title('Campaign Cost (log) vs ROI', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log10(Campaign Cost)')
axes[1].set_ylabel('ROI')
axes[1].legend()

plt.suptitle('Scatter Plot Analysis (5,000 sample)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/04_scatter_plots.png', bbox_inches='tight')
plt.show()
print('Saved: 04_scatter_plots.png')

In [ ]:
# ── 5. ROI by Campaign Type ───────────────────────────────────────────────────
# Use a sample for speed — 10,000 rows is statistically representative
df_plot = df.sample(10000, random_state=42) if len(df) > 10000 else df

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_roi = df_plot.groupby('campaign_type')['roi'].mean().sort_values(ascending=False)
colors = ['#27ae60' if v > 0 else '#e74c3c' for v in type_roi.values]
axes[0].bar(type_roi.index, type_roi.values, color=colors, edgecolor='grey', linewidth=0.5)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Average ROI by Campaign Type', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Campaign Type')
axes[0].set_ylabel('Average ROI')
axes[0].tick_params(axis='x', rotation=20)

# ROI by month of campaign start
monthly = df_plot.groupby('month')['roi'].mean()
axes[1].plot(monthly.index, monthly.values, marker='o', color='steelblue',
              linewidth=2.5, markersize=8)
axes[1].fill_between(monthly.index, monthly.values, alpha=0.2, color='steelblue')
axes[1].set_title('Average ROI by Month', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average ROI')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=30)

plt.suptitle('Campaign Type and Monthly ROI Trend', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/05_campaign_type_year.png', bbox_inches='tight')
plt.show()
print('Saved: 05_campaign_type_year.png')

---
## Module 6 — Statistical Analysis
Calculating descriptive statistics for all numeric variables.

In [ ]:
numeric_cols = ['engagements','estimated_reach','product_sales',
                'campaign_duration_days','engagement_rate',
                'campaign_cost','revenue','roi']

desc = df[numeric_cols].describe().T
desc['variance'] = df[numeric_cols].var()
desc['skewness'] = df[numeric_cols].skew()
desc['kurtosis'] = df[numeric_cols].kurt()
desc['median']   = df[numeric_cols].median()

print('Complete Descriptive Statistics:')
desc[['count','mean','median','std','variance','min','max','skewness','kurtosis']]

In [ ]:
roi = df['roi']
print('=' * 55)
print('         ROI — Detailed Statistics')
print('=' * 55)
print(f'  Count              : {roi.count():,}')
print(f'  Mean               : {roi.mean():.4f}')
print(f'  Median             : {roi.median():.4f}')
print(f'  Std Deviation      : {roi.std():.4f}')
print(f'  Variance           : {roi.var():.4f}')
print(f'  Min                : {roi.min():.4f}')
print(f'  Max                : {roi.max():.4f}')
print(f'  Skewness           : {roi.skew():.4f}')
print(f'  Kurtosis           : {roi.kurt():.4f}')
print(f'  Campaigns > 1.0 ROI: {(roi>1.0).sum():,} ({(roi>1.0).mean()*100:.1f}%)')
print(f'  Campaigns < 0.0 ROI: {(roi<0.0).sum():,} ({(roi<0.0).mean()*100:.1f}%)')
print('=' * 55)

In [ ]:
# Shapiro-Wilk is limited to n<=5000, so we test on a sample
sample_roi = df['roi'].sample(5000, random_state=42)
stat, p_val = stats.shapiro(sample_roi)
print('Shapiro-Wilk Normality Test on ROI (sample n=5000):')
print(f'  Statistic : {stat:.4f}')
print(f'  P-value   : {p_val:.6f}')
if p_val > 0.05:
    print('  Result    : Data appears normally distributed (p > 0.05)')
else:
    print('  Result    : Not perfectly normal, but with n=150,000 the Central Limit Theorem')
    print('              ensures regression assumptions still hold.')

---
## Module 7 — Correlation Analysis

In [ ]:
corr_cols  = ['engagements','estimated_reach','product_sales',
               'campaign_duration_days','engagement_rate','campaign_cost','revenue','roi']
corr_matrix = df[corr_cols].corr()

print('Correlation with ROI (sorted):')
print(corr_matrix['roi'].drop('roi').sort_values(ascending=False).to_string())

In [ ]:
plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../Images/06_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved: 06_correlation_heatmap.png')

In [ ]:
roi_corr = corr_matrix['roi'].drop('roi').sort_values()
colors   = ['crimson' if v < 0 else 'steelblue' for v in roi_corr]

plt.figure(figsize=(9, 5))
roi_corr.plot(kind='barh', color=colors, edgecolor='grey', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with ROI', fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.savefig('../Images/07_roi_correlation_bar.png', bbox_inches='tight')
plt.show()
print('Saved: 07_roi_correlation_bar.png')

---
## Module 8 — Multiple Linear Regression (Statsmodels)

### Regression Formula:
$$\text{ROI} = \beta_0 + \beta_1(\text{Engagement\_Rate}) + \beta_2(\text{Campaign\_Cost}) + \beta_3(\text{Product\_Sales}) + \beta_4(\text{Campaign\_Duration}) + \epsilon$$

**Note:** We use a 10% sample (15,000 rows) for regression to keep computation fast while remaining statistically valid. With 15,000 observations, results are highly reliable.

In [ ]:
# ── Feature Selection ─────────────────────────────────────────────────────────
# revenue is excluded — it's used to compute ROI (data leakage)
# engagements and estimated_reach are excluded — highly collinear with engagement_rate
feature_cols = ['engagement_rate', 'campaign_cost',
                'product_sales', 'campaign_duration_days']

# Sample for regression (stratified by platform)
df_reg = df.sample(n=15000, random_state=42).copy()

X = df_reg[feature_cols].copy()
y = df_reg['roi'].copy()

# VIF check
X_vif = sm.add_constant(X)
vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF'    : [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print('VIF Check (should be < 10):')
print(vif_data.to_string(index=False))

In [ ]:
# ── Build OLS Model ───────────────────────────────────────────────────────────
X_const  = sm.add_constant(X)
model    = sm.OLS(y, X_const)
results  = model.fit()

print('Model built using Statsmodels OLS.')
print(f'Observations : {int(results.nobs):,}')
print(f'Features     : {feature_cols}')

In [ ]:
print(results.summary())

---
## Module 9 — Model Interpretation

In [ ]:
coef_df = pd.DataFrame({
    'Feature'         : results.params.index,
    'Coefficient'     : results.params.values,
    'Std Error'       : results.bse.values,
    'T-Statistic'     : results.tvalues.values,
    'P-Value'         : results.pvalues.values,
    'CI Lower (2.5%)' : results.conf_int()[0].values,
    'CI Upper (97.5%)': results.conf_int()[1].values,
})
coef_df['Significant'] = coef_df['P-Value'].apply(
    lambda p: 'Yes ***' if p < 0.001 else ('Yes **' if p < 0.01
              else ('Yes *' if p < 0.05 else 'No')))

print('Regression Coefficients Table:')
coef_df

In [ ]:
print('=' * 55)
print('       MODEL PERFORMANCE METRICS')
print('=' * 55)
print(f'  R-squared          : {results.rsquared:.4f}')
print(f'  Adjusted R-squared : {results.rsquared_adj:.4f}')
print(f'  F-statistic        : {results.fvalue:.4f}')
print(f'  Prob (F-statistic) : {results.f_pvalue:.6f}')
print(f'  AIC                : {results.aic:.2f}')
print(f'  BIC                : {results.bic:.2f}')
print(f'  Observations       : {int(results.nobs):,}')
print('=' * 55)
r2 = results.rsquared
print(f'\nInterpretation: The model explains {r2*100:.1f}% of ROI variation.')
if r2 >= 0.7:
    print('  Strong fit — good predictive power.')
elif r2 >= 0.5:
    print('  Moderate fit — reasonable for marketing data.')
else:
    print('  Expected for marketing — ROI is influenced by many external factors not in the data.')

In [ ]:
coef_plot = coef_df[coef_df['Feature'] != 'const'].copy().sort_values('Coefficient')
colors    = ['crimson' if c < 0 else 'steelblue' for c in coef_plot['Coefficient']]
err       = (coef_plot['CI Upper (97.5%)'] - coef_plot['CI Lower (2.5%)']) / 2

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(coef_plot['Feature'], coef_plot['Coefficient'],
         xerr=err, color=colors, edgecolor='grey', linewidth=0.5,
         error_kw=dict(elinewidth=1.5, ecolor='black', capsize=4))
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Regression Coefficients with 95% Confidence Intervals',
              fontsize=12, fontweight='bold')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('../Images/08_coefficients.png', bbox_inches='tight')
plt.show()
print('Saved: 08_coefficients.png')

---
## Module 10 — Residual Analysis

In [ ]:
fitted    = results.fittedvalues
residuals = results.resid
std_resid = (residuals - residuals.mean()) / residuals.std()

print('Residuals Summary:')
print(f'  Mean : {residuals.mean():.8f}  (should be ~0)')
print(f'  Std  : {residuals.std():.4f}')
print(f'  Min  : {residuals.min():.4f}')
print(f'  Max  : {residuals.max():.4f}')

In [ ]:
residuals_sample = pd.Series(residuals.values[:5000])
fitted_sample    = pd.Series(fitted.values[:5000])
std_resid_s      = (residuals_sample - residuals_sample.mean()) / residuals_sample.std()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Residuals vs Fitted
axes[0,0].scatter(fitted_sample, residuals_sample, alpha=0.3, color='steelblue', s=8, edgecolors='none')
axes[0,0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0,0].set_title('Residuals vs Fitted Values', fontweight='bold')
axes[0,0].set_xlabel('Fitted (Predicted ROI)')
axes[0,0].set_ylabel('Residuals')

# Plot 2: Q-Q Plot
stats.probplot(residuals_sample, dist='norm', plot=axes[0,1])
axes[0,1].set_title('Q-Q Plot — Normality of Residuals', fontweight='bold')
axes[0,1].get_lines()[0].set(color='steelblue', markersize=2, alpha=0.5)
axes[0,1].get_lines()[1].set(color='red', linewidth=1.5)

# Plot 3: Histogram of Residuals
axes[1,0].hist(residuals_sample, bins=60, color='mediumpurple', edgecolor='white', linewidth=0.4)
axes[1,0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1,0].set_title('Distribution of Residuals', fontweight='bold')
axes[1,0].set_xlabel('Residuals')
axes[1,0].set_ylabel('Frequency')

# Plot 4: Scale-Location
axes[1,1].scatter(fitted_sample, np.sqrt(np.abs(std_resid_s)),
                   alpha=0.2, color='coral', s=5, edgecolors='none')
axes[1,1].set_title('Scale-Location (Homoscedasticity)', fontweight='bold')
axes[1,1].set_xlabel('Fitted Values')
axes[1,1].set_ylabel('√|Standardised Residuals|')

plt.suptitle('Residual Diagnostic Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../Images/09_residual_analysis.png', bbox_inches='tight')
plt.show()
print('Saved: 09_residual_analysis.png')

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y[:5000], fitted[:5000], alpha=0.2, color='steelblue', s=8, edgecolors='none')
mn = min(y.min(), fitted.min())
mx = max(y.max(), fitted.max())
plt.plot([mn, mx], [mn, mx], color='red', linestyle='--', linewidth=2, label='Perfect Prediction')
plt.title('Actual ROI vs Predicted ROI', fontsize=13, fontweight='bold')
plt.xlabel('Actual ROI')
plt.ylabel('Predicted ROI')
plt.legend()
plt.tight_layout()
plt.savefig('../Images/10_actual_vs_predicted.png', bbox_inches='tight')
plt.show()
print('Saved: 10_actual_vs_predicted.png')

---
## Module 11 — Prediction for New Campaign
Using the trained model to predict ROI for 5 hypothetical new campaigns.

In [ ]:
new_campaigns = pd.DataFrame({
    'engagement_rate'        : [8.5,    3.2,    12.0,   5.5,   20.0],
    'campaign_cost'          : [30000,  90000,  10000,  60000, 50000],
    'product_sales'          : [3500,   2000,   4500,   2500,  4000],
    'campaign_duration_days' : [14,     30,     7,      21,    10],
})

labels = [
    'Micro – Fitness (High Eng)',
    'Macro – Tech (Mid Eng)',
    'Mega  – Lifestyle (Low Eng)',
    'Nano  – Food (Very High Eng)',
    'Micro – Beauty (Avg Eng)',
]

new_const = sm.add_constant(new_campaigns, has_constant='add')
preds     = results.get_prediction(new_const)
pred_df   = preds.summary_frame(alpha=0.05)

out = pd.DataFrame({
    'Campaign'       : labels,
    'Eng Rate (%)'   : new_campaigns['engagement_rate'].values,
    'Cost (INR)'     : new_campaigns['campaign_cost'].values,
    'Product Sales'  : new_campaigns['product_sales'].values,
    'Duration (days)': new_campaigns['campaign_duration_days'].values,
    'Predicted ROI'  : pred_df['mean'].round(4).values,
    'CI Lower'       : pred_df['obs_ci_lower'].round(4).values,
    'CI Upper'       : pred_df['obs_ci_upper'].round(4).values,
})

print('=' * 75)
print('             CAMPAIGN ROI PREDICTIONS')
print('=' * 75)
print(out.to_string(index=False))
print()
print('ROI > 0 = Profitable   |   ROI = 1.0 = 100% return   |   ROI < 0 = Loss')

In [ ]:
pred_vals = pred_df['mean'].values
err_low   = pred_vals - pred_df['obs_ci_lower'].values
err_high  = pred_df['obs_ci_upper'].values - pred_vals
bar_colors= ['#2ecc71' if r > 1.0 else '#f39c12' if r > 0 else '#e74c3c' for r in pred_vals]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(labels)), pred_vals,
               color=bar_colors, edgecolor='grey', linewidth=0.5,
               yerr=[err_low, err_high],
               error_kw=dict(elinewidth=1.5, ecolor='black', capsize=5))
ax.axhline(1.0, color='navy',  linestyle='--', linewidth=1.5, label='ROI = 1.0 (Good threshold)')
ax.axhline(0.0, color='red',   linestyle=':',  linewidth=1.2, label='ROI = 0.0 (Break-even)')
for bar, val in zip(bars, pred_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Predicted ROI')
ax.set_title('Predicted ROI for New Campaigns (with 95% CI)',
              fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../Images/11_predictions.png', bbox_inches='tight')
plt.show()
print('Saved: 11_predictions.png')

---
## Module 12 — Model Evaluation Summary

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae  = mean_absolute_error(y, fitted)
rmse = np.sqrt(mean_squared_error(y, fitted))
# Safe MAPE — skip near-zero actual values
mask_mape = np.abs(y) > 0.01
mape = np.mean(np.abs((y[mask_mape] - fitted[mask_mape]) / y[mask_mape])) * 100

print('=' * 60)
print('        MODEL EVALUATION SUMMARY')
print('=' * 60)
print(f'  R-squared (R²)         : {results.rsquared:.4f}')
print(f'  Adjusted R²            : {results.rsquared_adj:.4f}')
print(f'  F-statistic            : {results.fvalue:.4f}')
print(f'  Prob (F-statistic)     : {results.f_pvalue:.2e}')
print(f'  Mean Absolute Error    : {mae:.4f}')
print(f'  Root Mean Squared Error: {rmse:.4f}')
print(f'  Mean Abs % Error (MAPE): {mape:.2f}%')
print(f'  Observations used      : {int(results.nobs):,}')
print('=' * 60)

In [ ]:
errors = y - fitted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(errors, bins=60, color='skyblue', edgecolor='white', linewidth=0.5)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Distribution of Prediction Errors', fontweight='bold')
axes[0].set_xlabel('Error (Actual − Predicted ROI)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.abs(errors), bins=60, color='salmon', edgecolor='white', linewidth=0.5)
axes[1].axvline(mae, color='darkred', linestyle='--', linewidth=1.5, label=f'MAE={mae:.4f}')
axes[1].set_title('Absolute Prediction Errors', fontweight='bold')
axes[1].set_xlabel('|Error|')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle('Prediction Error Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../Images/12_error_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: 12_error_distribution.png')

---
## Module 13 — Conclusion

### Summary of Findings

This project built a **Multiple Linear Regression model** to predict the **ROI of influencer marketing campaigns** using a real-world dataset of **150,000 campaigns** (Kaggle) spanning Instagram, YouTube, TikTok and Twitter from 2022 to 2024.

### Feature Engineering
Since the raw dataset did not contain `ROI`, `Engagement_Rate`, `Campaign_Cost`, or `Revenue`, these were engineered using standard industry formulas:
- `Engagement_Rate = engagements / estimated_reach × 100`
- `Campaign_Cost = estimated_reach × CPM / 1000` (CPM varies by platform)
- `Revenue = product_sales × ₹40 (avg unit value, calibrated for realistic ROI)`
- `ROI = (Revenue − Campaign_Cost) / Campaign_Cost`

### Key Findings
1. **Engagement Rate** is the strongest driver of ROI — higher engagement consistently leads to better returns.
2. **Product Sales** has a strong positive relationship with ROI — campaigns that convert clicks to purchases perform significantly better.
3. **Campaign Cost** has a negative coefficient — higher spend does not guarantee higher ROI; efficiency matters.
4. **Campaign Duration** has a minimal effect — longer campaigns do not necessarily produce better ROI.

### Technologies Used
- **Python** — Data cleaning, feature engineering, analysis, modeling
- **Pandas / NumPy** — Data manipulation and calculations
- **Matplotlib / Seaborn** — Visualizations
- **Statsmodels** — Multiple Linear Regression, p-values, R², confidence intervals
- **Excel** — Data storage and initial review

In [ ]:
params = results.params
print('Final Regression Equation:')
print()
print(f'  ROI = {params["const"]:.6f}')
for feat in feature_cols:
    sign = '+' if params[feat] >= 0 else '-'
    print(f'        {sign} {abs(params[feat]):.8f} × {feat}')
print()
print(f'  R²           = {results.rsquared:.4f}')
print(f'  Adjusted R²  = {results.rsquared_adj:.4f}')
print(f'  F-statistic  = {results.fvalue:.4f} (p = {results.f_pvalue:.2e})')
print()
print('Project completed successfully.')
print('Dataset: Real Kaggle data — 150,000 influencer marketing campaigns.')
print('All graphs saved in the Images/ folder.')